# 딥페이크 감지 모델: Vision Transformer를 이용한 접근

이 노트북은 Celeb-DF 데이터셋을 사용하여 Vision Transformer 기반의 딥페이크 감지 모델을 학습합니다.

## 대회 정보
- **대회**: 딥페이크 범죄 대응을 위한 AI 탐지 모델 경진대회
- **평가지표**: Macro F1-Score
- **데이터**: Celeb-DF v2 (5,639 fake + 590 real videos)

## 모델 아키텍처
- Vision Transformer (ViT) 기반
- 임베딩 벡터를 통한 특징 추출
- 멀티모달 접근 (비디오 → 프레임 → 얼굴 → 특징)

## 목차
1. 환경 설정 및 라이브러리 임포트
2. 데이터셋 다운로드 및 준비
3. 데이터 전처리 (얼굴 검출, 프레임 추출)
4. Vision Transformer 모델 구축
5. 학습 루프 구현
6. 평가 및 Macro F1-Score 계산
7. 추론 및 제출 파일 생성

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
# 필요한 패키지 설치
! pip install torch torchvision timm transformers
! pip install opencv-python face-recognition dlib
! pip install decord albumentations
! pip install scikit-learn pandas numpy matplotlib seaborn tqdm

In [ ]:
import os
import json
import random
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Computer Vision
import cv2
import face_recognition
from decord import VideoReader, cpu

# Vision Transformer
import timm
from transformers import ViTModel, ViTConfig

# Data Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# 시드 고정
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 데이터셋 다운로드 및 준비

### Celeb-DF 데이터셋 구조
```
Celeb-DF/
├── Celeb-real/        # 590 real celebrity videos
├── YouTube-real/      # 300 real YouTube videos
├── Celeb-synthesis/   # 5,639 deepfake videos
└── List_of_testing_videos.txt  # 518 test videos
```

**참고**: Celeb-DF 데이터셋은 [Google Form](https://forms.gle/...) 작성 후 승인을 받아야 다운로드할 수 있습니다.

In [ ]:
# 데이터셋 경로 설정
DATA_ROOT = Path('./Celeb-DF')  # 다운로드 받은 데이터셋 경로
REAL_DIR = DATA_ROOT / 'Celeb-real'
YOUTUBE_DIR = DATA_ROOT / 'YouTube-real'
FAKE_DIR = DATA_ROOT / 'Celeb-synthesis'
TEST_LIST = DATA_ROOT / 'List_of_testing_videos.txt'

# 출력 디렉토리
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(exist_ok=True)

# 전처리된 데이터 저장 경로
PROCESSED_DIR = Path('./processed_data')
PROCESSED_DIR.mkdir(exist_ok=True)

In [ ]:
# 데이터셋 파일 목록 생성
def get_video_paths(data_root: Path) -> Dict[str, List[Path]]:
    """
    Celeb-DF 데이터셋의 비디오 파일 경로를 수집합니다.
    
    Returns:
        Dict with 'real' and 'fake' video paths
    """
    video_paths = {
        'real': [],
        'fake': []
    }
    
    # Real videos from Celeb-real and YouTube-real
    real_dir = data_root / 'Celeb-real'
    youtube_dir = data_root / 'YouTube-real'
    
    if real_dir.exists():
        video_paths['real'].extend(list(real_dir.glob('*.mp4')))
    if youtube_dir.exists():
        video_paths['real'].extend(list(youtube_dir.glob('*.mp4')))
    
    # Fake videos from Celeb-synthesis
    fake_dir = data_root / 'Celeb-synthesis'
    if fake_dir.exists():
        video_paths['fake'].extend(list(fake_dir.glob('*.mp4')))
    
    print(f"Real videos: {len(video_paths['real'])}")
    print(f"Fake videos: {len(video_paths['fake'])}")
    
    return video_paths

# 데이터셋 로드
video_paths = get_video_paths(DATA_ROOT)

## 3. 데이터 전처리

### 전처리 파이프라인
1. **비디오 → 프레임 추출**: 각 비디오에서 N개의 균일 분포 프레임 추출
2. **얼굴 검출**: 각 프레임에서 얼굴 영역 검출 (face_recognition 또는 dlib 사용)
3. **얼굴 정렬 및 크롭**: 얼굴 영역만 추출하여 224x224로 리사이즈
4. **정규화**: ImageNet 평균/표준편차로 정규화
5. **임베딩 추출**: ViT를 통해 임베딩 벡터 생성

In [ ]:
class FaceExtractor:
    """
    비디오에서 얼굴을 추출하는 클래스
    """
    def __init__(self, face_size=(224, 224)):
        self.face_size = face_size
    
    def extract_frames(self, video_path: Path, num_frames: int = 16) -> List[np.ndarray]:
        """
        비디오에서 균일하게 분포된 프레임을 추출합니다.
        
        Args:
            video_path: 비디오 파일 경로
            num_frames: 추출할 프레임 수
        
        Returns:
            List of frames (numpy arrays)
        """
        try:
            vr = VideoReader(str(video_path), ctx=cpu(0))
            total_frames = len(vr)
            
            # 균일하게 분포된 프레임 인덱스 계산
            if total_frames < num_frames:
                indices = list(range(total_frames))
            else:
                indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
            
            frames = []
            for idx in indices:
                frame = vr[idx].asnumpy()
                frames.append(frame)
            
            return frames
        
        except Exception as e:
            print(f"Error extracting frames from {video_path}: {e}")
            return []
    
    def detect_and_crop_face(self, frame: np.ndarray) -> np.ndarray:
        """
        프레임에서 얼굴을 검출하고 크롭합니다.
        
        Args:
            frame: RGB 이미지 (H, W, 3)
        
        Returns:
            Cropped face image or original frame if no face detected
        """
        # face_recognition 라이브러리 사용
        face_locations = face_recognition.face_locations(frame)
        
        if len(face_locations) > 0:
            # 첫 번째 얼굴 사용 (대회 규격: 1명만 포함)
            top, right, bottom, left = face_locations[0]
            
            # 얼굴 영역에 여유 추가 (1.3배 확대)
            h, w = bottom - top, right - left
            center_y, center_x = (top + bottom) // 2, (left + right) // 2
            size = int(max(h, w) * 1.3)
            
            top = max(0, center_y - size // 2)
            bottom = min(frame.shape[0], center_y + size // 2)
            left = max(0, center_x - size // 2)
            right = min(frame.shape[1], center_x + size // 2)
            
            face = frame[top:bottom, left:right]
            
            # 리사이즈
            face = cv2.resize(face, self.face_size)
            return face
        
        else:
            # 얼굴을 찾지 못한 경우 전체 프레임 사용
            return cv2.resize(frame, self.face_size)
    
    def process_video(self, video_path: Path, num_frames: int = 16) -> List[np.ndarray]:
        """
        비디오를 처리하여 얼굴 이미지 리스트를 반환합니다.
        
        Args:
            video_path: 비디오 파일 경로
            num_frames: 추출할 프레임 수
        
        Returns:
            List of face images
        """
        frames = self.extract_frames(video_path, num_frames)
        faces = []
        
        for frame in frames:
            face = self.detect_and_crop_face(frame)
            faces.append(face)
        
        return faces

In [ ]:
# 데이터 증강 (Albumentations)
def get_train_transforms():
    return A.Compose([
        A.Resize(224, 224),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.2),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(224, 224),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2()
    ])

In [ ]:
class DeepfakeDataset(Dataset):
    """
    딥페이크 감지를 위한 커스텀 데이터셋
    """
    def __init__(
        self,
        video_paths: List[Tuple[Path, int]],
        face_extractor: FaceExtractor,
        transform=None,
        num_frames: int = 16,
        frames_per_video: int = 8
    ):
        """
        Args:
            video_paths: List of (video_path, label) tuples
            face_extractor: FaceExtractor instance
            transform: Albumentations transforms
            num_frames: Number of frames to extract from video
            frames_per_video: Number of random frames to use per video
        """
        self.video_paths = video_paths
        self.face_extractor = face_extractor
        self.transform = transform
        self.num_frames = num_frames
        self.frames_per_video = frames_per_video
    
    def __len__(self):
        return len(self.video_paths)
    
    def __getitem__(self, idx):
        video_path, label = self.video_paths[idx]
        
        # 비디오에서 얼굴 추출
        faces = self.face_extractor.process_video(video_path, self.num_frames)
        
        if len(faces) == 0:
            # 얼굴을 찾지 못한 경우 빈 이미지 반환
            faces = [np.zeros((224, 224, 3), dtype=np.uint8)]
        
        # 랜덤하게 frames_per_video개 선택
        if len(faces) > self.frames_per_video:
            indices = random.sample(range(len(faces)), self.frames_per_video)
            faces = [faces[i] for i in indices]
        
        # 변환 적용
        transformed_faces = []
        for face in faces:
            if self.transform:
                augmented = self.transform(image=face)
                transformed_faces.append(augmented['image'])
        
        # 프레임들을 스택 (frames_per_video, 3, 224, 224)
        frames_tensor = torch.stack(transformed_faces)
        
        return frames_tensor, torch.tensor(label, dtype=torch.long)

In [ ]:
# 데이터셋 분할
def prepare_dataset(video_paths: Dict[str, List[Path]], test_size: float = 0.2, val_size: float = 0.1):
    """
    데이터셋을 train, validation, test로 분할합니다.
    
    Args:
        video_paths: Dict with 'real' and 'fake' video paths
        test_size: Test set ratio
        val_size: Validation set ratio (from train set)
    
    Returns:
        train_data, val_data, test_data: List of (path, label) tuples
    """
    # 라벨 생성 (Real: 0, Fake: 1)
    real_data = [(path, 0) for path in video_paths['real']]
    fake_data = [(path, 1) for path in video_paths['fake']]
    
    all_data = real_data + fake_data
    
    # Train / Test 분할
    train_val_data, test_data = train_test_split(
        all_data,
        test_size=test_size,
        random_state=42,
        stratify=[label for _, label in all_data]
    )
    
    # Train / Validation 분할
    train_data, val_data = train_test_split(
        train_val_data,
        test_size=val_size,
        random_state=42,
        stratify=[label for _, label in train_val_data]
    )
    
    print(f"Train: {len(train_data)} ({sum(l for _, l in train_data)} fake)")
    print(f"Val: {len(val_data)} ({sum(l for _, l in val_data)} fake)")
    print(f"Test: {len(test_data)} ({sum(l for _, l in test_data)} fake)")
    
    return train_data, val_data, test_data

# 데이터 분할
train_data, val_data, test_data = prepare_dataset(video_paths)

## 4. Vision Transformer 모델 구축

### 모델 아키텍처
1. **ViT Backbone**: 사전학습된 ViT 모델 (timm 라이브러리 사용)
2. **Temporal Aggregation**: 비디오의 여러 프레임을 통합 (평균 풀링 또는 어텐션)
3. **Classification Head**: 이진 분류를 위한 FC 레이어

In [ ]:
class ViTDeepfakeDetector(nn.Module):
    """
    Vision Transformer 기반 딥페이크 감지 모델
    """
    def __init__(
        self,
        model_name: str = 'vit_base_patch16_224',
        pretrained: bool = True,
        num_classes: int = 2,
        dropout: float = 0.3
    ):
        super(ViTDeepfakeDetector, self).__init__()
        
        # ViT Backbone (timm 라이브러리)
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0  # Remove classification head
        )
        
        # 임베딩 차원
        self.embedding_dim = self.backbone.num_features
        
        # Temporal Attention (여러 프레임의 정보를 통합)
        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=self.embedding_dim,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        # Classification Head
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.embedding_dim),
            nn.Dropout(dropout),
            nn.Linear(self.embedding_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        """
        Forward pass
        
        Args:
            x: (batch_size, num_frames, 3, 224, 224)
        
        Returns:
            logits: (batch_size, num_classes)
        """
        batch_size, num_frames, c, h, w = x.shape
        
        # 프레임별로 ViT 특징 추출
        # (batch_size * num_frames, 3, 224, 224)
        x = x.view(batch_size * num_frames, c, h, w)
        
        # ViT를 통한 임베딩 추출
        # (batch_size * num_frames, embedding_dim)
        frame_embeddings = self.backbone(x)
        
        # (batch_size, num_frames, embedding_dim)
        frame_embeddings = frame_embeddings.view(batch_size, num_frames, -1)
        
        # Temporal Attention으로 프레임 정보 통합
        # (batch_size, num_frames, embedding_dim)
        attn_output, _ = self.temporal_attention(
            frame_embeddings,
            frame_embeddings,
            frame_embeddings
        )
        
        # 평균 풀링으로 비디오 레벨 임베딩 생성
        # (batch_size, embedding_dim)
        video_embedding = attn_output.mean(dim=1)
        
        # 분류
        # (batch_size, num_classes)
        logits = self.classifier(video_embedding)
        
        return logits

# 모델 초기화
model = ViTDeepfakeDetector(
    model_name='vit_base_patch16_224',
    pretrained=True,
    num_classes=2,
    dropout=0.3
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. 학습 루프 구현

### 학습 설정
- **Optimizer**: AdamW
- **Scheduler**: CosineAnnealingLR with Warmup
- **Loss**: CrossEntropyLoss (클래스 가중치 적용 가능)
- **Metrics**: Macro F1-Score, Accuracy

In [ ]:
# 하이퍼파라미터
BATCH_SIZE = 4  # 비디오 데이터이므로 작은 배치 크기
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_FRAMES = 16
FRAMES_PER_VIDEO = 8

# Face Extractor
face_extractor = FaceExtractor(face_size=(224, 224))

# 데이터셋 및 데이터로더
train_dataset = DeepfakeDataset(
    train_data,
    face_extractor,
    transform=get_train_transforms(),
    num_frames=NUM_FRAMES,
    frames_per_video=FRAMES_PER_VIDEO
)

val_dataset = DeepfakeDataset(
    val_data,
    face_extractor,
    transform=get_val_transforms(),
    num_frames=NUM_FRAMES,
    frames_per_video=FRAMES_PER_VIDEO
)

test_dataset = DeepfakeDataset(
    test_data,
    face_extractor,
    transform=get_val_transforms(),
    num_frames=NUM_FRAMES,
    frames_per_video=FRAMES_PER_VIDEO
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
# Loss Function (클래스 불균형 고려)
# Celeb-DF는 fake가 훨씬 많으므로 가중치 조정
num_real = sum(1 for _, label in train_data if label == 0)
num_fake = sum(1 for _, label in train_data if label == 1)
class_weights = torch.tensor([num_fake / num_real, 1.0]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

In [ ]:
def calculate_macro_f1(y_true, y_pred):
    """
    Macro F1-Score 계산 (대회 평가지표)
    
    Args:
        y_true: Ground truth labels
        y_pred: Predicted labels
    
    Returns:
        macro_f1: Macro F1-Score
    """
    return f1_score(y_true, y_pred, average='macro')

def train_one_epoch(model, loader, criterion, optimizer, device):
    """
    1 에폭 학습
    """
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training')
    for frames, labels in pbar:
        frames = frames.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward
        logits = model(frames)
        loss = criterion(logits, labels)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        # Metrics
        running_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': loss.item()})
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = calculate_macro_f1(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, epoch_f1

def validate(model, loader, criterion, device):
    """
    검증 및 평가
    """
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for frames, labels in pbar:
            frames = frames.to(device)
            labels = labels.to(device)
            
            # Forward
            logits = model(frames)
            loss = criterion(logits, labels)
            
            # Metrics
            running_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': loss.item()})
    
    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = calculate_macro_f1(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels

In [ ]:
# 학습 루프
best_val_f1 = 0.0
history = {
    'train_loss': [],
    'train_acc': [],
    'train_f1': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': []
}

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    print("-" * 50)
    
    # 학습
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # 검증
    val_loss, val_acc, val_f1, _, _ = validate(
        model, val_loader, criterion, device
    )
    
    # Scheduler 업데이트
    scheduler.step()
    
    # 결과 출력
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1 (Macro): {val_f1:.4f}")
    
    # 히스토리 저장
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    # 최고 모델 저장
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pth')
        print(f"✅ Best model saved! Val F1: {best_val_f1:.4f}")

print("\n🎉 Training completed!")
print(f"Best Validation Macro F1-Score: {best_val_f1:.4f}")

## 6. 평가 및 결과 시각화

In [ ]:
# 학습 곡선 시각화
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

# Macro F1-Score
axes[2].plot(history['train_f1'], label='Train F1', marker='o')
axes[2].plot(history['val_f1'], label='Val F1 (Macro)', marker='s')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Macro F1-Score')
axes[2].set_title('Training and Validation Macro F1-Score')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=300)
plt.show()

In [ ]:
# 최고 모델 로드
model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pth'))

# 테스트 평가
test_loss, test_acc, test_f1, test_preds, test_labels = validate(
    model, test_loader, criterion, device
)

print("\n" + "="*50)
print("📊 Test Set Evaluation")
print("="*50)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro F1-Score: {test_f1:.4f}")
print("="*50)

# Classification Report
print("\nClassification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=['Real', 'Fake']
))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Real', 'Fake'],
    yticklabels=['Real', 'Fake']
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig(OUTPUT_DIR / 'confusion_matrix.png', dpi=300)
plt.show()

## 7. 추론 및 제출 파일 생성

대회 제출용 추론 코드입니다.

In [ ]:
def predict_video(model, video_path: Path, face_extractor, transform, device, num_frames=16):
    """
    단일 비디오에 대한 예측
    
    Args:
        model: Trained model
        video_path: Path to video file
        face_extractor: FaceExtractor instance
        transform: Transforms
        device: Device
        num_frames: Number of frames to extract
    
    Returns:
        prediction: 0 (Real) or 1 (Fake)
        confidence: Confidence score
    """
    model.eval()
    
    # 얼굴 추출
    faces = face_extractor.process_video(video_path, num_frames)
    
    if len(faces) == 0:
        # 얼굴을 찾지 못한 경우 기본값
        return 0, 0.5
    
    # 변환 적용
    transformed_faces = []
    for face in faces:
        augmented = transform(image=face)
        transformed_faces.append(augmented['image'])
    
    # 텐서로 변환 (1, num_frames, 3, 224, 224)
    frames_tensor = torch.stack(transformed_faces).unsqueeze(0).to(device)
    
    # 예측
    with torch.no_grad():
        logits = model(frames_tensor)
        probs = torch.softmax(logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        conf = probs[0, pred].item()
    
    return pred, conf

In [ ]:
# 테스트 데이터 예측 예시
sample_video_path = test_data[0][0]  # 첫 번째 테스트 비디오
true_label = test_data[0][1]

pred, conf = predict_video(
    model,
    sample_video_path,
    face_extractor,
    get_val_transforms(),
    device
)

print(f"Video: {sample_video_path.name}")
print(f"True Label: {'Real' if true_label == 0 else 'Fake'}")
print(f"Prediction: {'Real' if pred == 0 else 'Fake'}")
print(f"Confidence: {conf:.4f}")

In [ ]:
# 제출 파일 생성 (대회 형식에 맞게 조정 필요)
def create_submission(model, test_video_paths, face_extractor, transform, device, output_path):
    """
    제출용 CSV 파일 생성
    
    Args:
        model: Trained model
        test_video_paths: List of test video paths
        face_extractor: FaceExtractor instance
        transform: Transforms
        device: Device
        output_path: Output CSV path
    """
    results = []
    
    for video_path in tqdm(test_video_paths, desc='Creating submission'):
        pred, conf = predict_video(
            model, video_path, face_extractor, transform, device
        )
        
        results.append({
            'filename': video_path.name,
            'prediction': pred,  # 0: Real, 1: Fake
            'confidence': conf
        })
    
    # DataFrame 생성
    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False)
    print(f"✅ Submission file saved to {output_path}")
    
    return df

# 제출 파일 생성 예시
# test_video_paths = [path for path, _ in test_data]
# submission_df = create_submission(
#     model,
#     test_video_paths,
#     face_extractor,
#     get_val_transforms(),
#     device,
#     OUTPUT_DIR / 'submission.csv'
# )

## 추가 개선 방안

### 1. 모델 앙상블
- 여러 ViT 변형 (ViT-B, ViT-L, Swin Transformer 등) 앙상블
- CNN 모델 (EfficientNet, XceptionNet)과 결합

### 2. 데이터 증강 강화
- Cutout, Mixup, CutMix
- Temporal augmentation (프레임 순서 변경, 속도 조절)

### 3. 고급 특징 추출
- Frequency domain analysis (FFT, DCT)
- Optical flow 정보 활용
- 얼굴 랜드마크 일관성 검사

### 4. 추가 데이터셋
- FaceForensics++, DFDC와 혼합 학습
- 외부 데이터 활용 (규칙 확인 필요)

### 5. 학습 전략
- Focal Loss로 어려운 샘플에 집중
- Pseudo labeling
- Self-supervised pre-training

### 6. 후처리
- Test-Time Augmentation (TTA)
- 비디오 레벨 평활화 (temporal smoothing)

## 참고 자료

### 논문
- [Celeb-DF Paper (CVPR 2020)](https://openaccess.thecvf.com/content_CVPR_2020/papers/Li_Celeb-DF_A_Large-Scale_Challenging_Dataset_for_DeepFake_Forensics_CVPR_2020_paper.pdf)
- [Vision Transformer (ViT)](https://arxiv.org/abs/2010.11929)
- [A Timely Survey on Vision Transformer for Deepfake Detection](https://arxiv.org/abs/2405.08463)

### GitHub
- [Celeb-DF Dataset](https://github.com/yuezunli/celeb-deepfakeforensics)
- [GenConViT](https://github.com/erprogs/GenConViT)
- [CViT](https://github.com/erprogs/CViT)

### 라이브러리
- [timm (PyTorch Image Models)](https://github.com/huggingface/pytorch-image-models)
- [Transformers](https://github.com/huggingface/transformers)
- [Albumentations](https://github.com/albumentations-team/albumentations)